# Treatment-Aware Telecom Churn Targeting

This notebook is a compact, reproducible exploration of the Orange Belgium / ULB randomized retention dataset. Production logic lives in `src/`; the notebook imports it instead of duplicating preprocessing or model evaluation.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from src.config import DEFAULT_DATA_PATH, METRICS_DIR
from src.data_preprocessing import clean_data, get_feature_types, load_data, split_features_target

sns.set_theme(style='whitegrid')

## 1. Load and validate data

If the file is absent, run `python -m src.download_data` from the repository root. Loading validates the official schema before analysis.

In [ ]:
df = clean_data(load_data(DEFAULT_DATA_PATH))
X, y, treatment = split_features_target(df)
numeric_features, categorical_features = get_feature_types(X)

pd.Series({
    'rows': len(df),
    'features': X.shape[1],
    'numeric_features': len(numeric_features),
    'categorical_features': len(categorical_features),
    'churn_rate': y.mean(),
    'treatment_rate': treatment.mean(),
})

## 2. Randomized treatment and observed outcome

The raw difference below is the average intent-to-treat effect in the sample. It does not reveal heterogeneous treatment response, which is the modeling target.

In [ ]:
outcomes = pd.DataFrame({'treatment': treatment, 'churn': y})
rates = outcomes.groupby('treatment')['churn'].agg(['mean', 'count', 'sum'])
rates.index = ['Control', 'Treated']
display(rates)
print(f"Observed churn reduction: {rates.loc['Control', 'mean'] - rates.loc['Treated', 'mean']:.3%}")

ax = rates['mean'].plot(kind='bar', rot=0, title='Observed churn by randomized treatment')
ax.set_ylabel('Churn rate')
plt.show()

## 3. Anonymized feature audit

Numerical columns are PCA components and categorical columns are anonymized factors. We inspect distributions and cardinality but do not invent business meanings.

In [ ]:
display(X[numeric_features[:10]].describe().T)
display(X[categorical_features].nunique().sort_values().rename('cardinality').to_frame())

fig, axes = plt.subplots(2, 2, figsize=(11, 7))
for column, ax in zip(numeric_features[:4], axes.ravel()):
    sns.histplot(data=df, x=column, hue='y', bins=30, stat='density', common_norm=False, ax=ax)
    ax.set_title(f'{column} by churn outcome')
fig.tight_layout()
plt.show()

## 4. Train without test leakage

The script reserves holdout first, creates OOF predictions, selects the model and campaign size from OOF only, and touches holdout once.

In [ ]:
# Run from a terminal at repository root:
# python -m src.train

comparison_path = METRICS_DIR / 'model_comparison.csv'
if comparison_path.exists():
    display(pd.read_csv(comparison_path))
else:
    print('Training artifacts are not present yet. Run: python -m src.train')

## 5. Interpretation guardrails

- AUUC/Qini evaluate ranking by incremental outcome, not ordinary classification.
- A wide bootstrap interval means the treatment policy is uncertain.
- Model feature importance is not a causal explanation.
- A new randomized campaign is required before operational rollout.